## Setup and inputs

In [ ]:
# 1. Setup: imports and paths
import os, json, time, pickle, datetime, contextlib, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

N = 5000                                              
CLASSES = ["AFib", "GSVT", "SB", "SR"]

ECG_ROOT = "/kaggle/input/datasets/aihash4/ecgdata/ECGData"
IMG_ROOT = "/kaggle/input/datasets/fatmamurat/imagedata-label2/label1"
MODEL1D  = "/kaggle/input/datasets/aihash4/keras1dand2d/MMGradCAM_1d_trained_model.keras"
MODEL2D  = "/kaggle/input/datasets/aihash4/keras1dand2d/MMGradCAM_2d_trained_model.keras"
REPO_URL = "https://github.com/dwiharyani/dan-mmgradcam-anchor-replication.git"
REPO_DIR = "/kaggle/working/dan-mmgradcam-anchor-replication"
OUT      = "/kaggle/working/m2_out"
SCALER_PATH = "/kaggle/working/standard_scaler.npz"
os.makedirs(OUT, exist_ok=True)

for p in [ECG_ROOT, IMG_ROOT, MODEL1D, MODEL2D]:
    print("OK     " if os.path.exists(p) else "MISSING", p)
print("TensorFlow", tf.__version__, "| GPUs:", [d.name for d in tf.config.list_physical_devices("GPU")])


In [ ]:
# 2. Repo + test manifest
if not os.path.exists(REPO_DIR):
    r = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)
    print("clone ok" if r.returncode == 0 else r.stderr)
else:
    print("repo already present")

test = pd.read_csv(f"{REPO_DIR}/reproducibility/test_split_1054.csv")
ecg_files = sorted(f for f in os.listdir(ECG_ROOT) if f.lower().endswith(".csv"))

assert len(test) == 1054 and test["cohort_index"].is_unique and test["ecg_csv_filename"].is_unique
missing = set(test["ecg_csv_filename"]) - set(ecg_files)
assert not missing, f"{len(missing)} test ECG files missing"
print("ECG CSV files in folder:", len(ecg_files), "| test records:", len(test))
print(test["class_name"].value_counts().to_dict())
print("test columns:", test.columns.tolist())


In [ ]:
# 3. ECG loader (Lead II) + check against the signals saved by your earlier run
def load_ecg(fn):
    a = pd.read_csv(os.path.join(ECG_ROOT, fn), usecols=["II"])["II"].to_numpy(dtype=np.float64)
    assert a.shape == (N,) and np.isfinite(a).all(), (fn, a.shape)
    return a

SAVED_TEST = "/kaggle/working/X_test_raw.npy"          
if os.path.exists(SAVED_TEST):
    Xt_saved = np.load(SAVED_TEST, mmap_mode="r")
    assert Xt_saved.shape == (len(test), N), Xt_saved.shape
    idx = np.linspace(0, len(test) - 1, 25).astype(int)
    d = max(float(np.abs(load_ecg(test["ecg_csv_filename"].iloc[i]) - Xt_saved[i]).max()) for i in idx)
    print("max |diff| vs saved test signals (25 records):", d)
    assert d < 1e-2, "load_ecg does NOT reproduce the signals used earlier - stop and tell Claude"
    print("load_ecg reproduces the earlier signals")
else:
    x = load_ecg(test["ecg_csv_filename"].iloc[0])
    print("No saved test signals to compare with; shape check only:", x.shape)


In [ ]:
# 4. 1D scaler (PROVISIONAL: fitted on 9,592 records, intended cohort is 9,534)

REFIT_SCALER = False
if REFIT_SCALER or not os.path.exists(SCALER_PATH):
    from joblib import Parallel, delayed
    from tqdm.auto import tqdm
    fit_files = sorted(set(ecg_files) - set(test["ecg_csv_filename"]))
    assert len(fit_files) == 9592, len(fit_files)       
    X = np.stack(Parallel(n_jobs=-1, batch_size=64)(delayed(load_ecg)(f) for f in tqdm(fit_files)))
    mean, var = X.mean(0), X.var(0)
    scale = np.sqrt(var); scale[scale == 0] = 1.0
    np.savez(SCALER_PATH, mean=mean, scale=scale, var=var)
    print("REFIT scaler on", X.shape[0], "records (provisional)")
    del X
sc = np.load(SCALER_PATH)
MEAN, SCALE = sc["mean"], sc["scale"]
assert MEAN.shape == (N,) and SCALE.shape == (N,)
SCALER_N_FIT = 9592
print("scaler loaded:", MEAN.shape, "| PROVISIONAL - fitted on", SCALER_N_FIT, "records, intended 9,534")


In [ ]:
# 6. Pretrained models (not retrained)
def last_conv(model, layer_type):
    return next(l.name for l in reversed(model.layers) if isinstance(l, layer_type))

m1 = tf.keras.models.load_model(MODEL1D)
m2 = tf.keras.models.load_model(MODEL2D)
L1 = last_conv(m1, tf.keras.layers.Conv1D)
L2 = last_conv(m2, tf.keras.layers.Conv2D)
print("1D:", m1.input_shape, "->", m1.output_shape, "| last Conv1D:", L1)
print("2D:", m2.input_shape, "->", m2.output_shape, "| last Conv2D:", L2)
assert m1.input_shape[1:] == (N, 1) and m2.input_shape[1:] == (300, 600, 1)


## Module 2: perturbation and stability experiment

In [ ]:
# 7. MODULE 2 v2 definitions (no computation happens in this cell)

import os, json, time, pickle, datetime, contextlib 
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import maximum_filter1d

_missing = [n for n in ["N", "load_ecg", "test", "OUT", "MEAN", "SCALE", "SCALER_N_FIT",
                       "m1", "m2", "L1", "L2", "last_conv"] if n not in globals()]
assert not _missing, f"Run the earlier cells first. Not defined: {_missing}"
assert N == 5000 and len(test) == 1054
assert test["ecg_csv_filename"].is_unique and test["cohort_index"].is_unique

EPS, FS, DRIFT_HZ = 1e-8, 500.0, 0.3
CLASSES = ["AFib", "GSVT", "SB", "SR"]
SCALES = [0.8, 0.9, 0.95, 1.05, 1.1, 1.2]
DRIFTS = [0.05, 0.10]
DRIFT_PHASES = np.linspace(0, 2 * np.pi, 6, endpoint=False)
NOISES = [0.05, 0.10]
NOISE_SEEDS = 6
N_PERT = 30
assert len(SCALES) + len(DRIFTS) * len(DRIFT_PHASES) + len(NOISES) * NOISE_SEEDS == N_PERT

X0, X1, W = 58, 1021, 600        
MASK_DILATE_Y, COVERAGE_MIN = 6, 0.75
MIN_ACCEPTED, CKPT_EVERY = 2, 25
FAMILIES = ["scale"] + [f"drift_{a}" for a in DRIFTS] + [f"noise_{a}" for a in NOISES]
BS = {"1d": 32, "2d": 8}
FORCE_CPU = {"1d": False, "2d": False}

CONFIG = json.loads(json.dumps(dict(
    version="v2-module1-aligned", FS=FS, DRIFT_HZ=DRIFT_HZ, SCALES=SCALES, DRIFTS=DRIFTS,
    DRIFT_PHASES=DRIFT_PHASES.tolist(), NOISES=NOISES, NOISE_SEEDS=NOISE_SEEDS,
    X0=X0, X1=X1, W=W, MASK_DILATE_Y=MASK_DILATE_Y, COVERAGE_MIN=COVERAGE_MIN,
    MIN_ACCEPTED=MIN_ACCEPTED)))

def grayscale_norm(values, valid):
    a = np.asarray(values, dtype=np.float64)
    if not np.isfinite(a).all():
        raise ValueError("Non-finite attribution")
    subset = a[valid]
    if subset.size == 0:
        raise ValueError("No valid temporal columns")
    lo, hi = float(subset.min()), float(subset.max())
    if hi - lo <= 1e-10:
        return None
    out = np.zeros(a.shape, dtype=np.float32)
    out[valid] = ((subset - lo) / (hi - lo)).astype(np.float32)
    return out

def trace_mask_at_2d_grid(original_rgb, target_h=300, target_w=600, dilate_y=6):
    im = np.asarray(original_rgb, dtype=np.int16)
    R, G, B = im[..., 0], im[..., 1], im[..., 2]
    ink = ((B - R >= 14) & (B - G >= 5) & (G - R >= 4) &
           (R < 240) & (G < 250))
    resized = Image.fromarray(ink.astype(np.uint8) * 255, 'L').resize(
        (target_w, target_h), Image.Resampling.BILINEAR)
    mask = (np.asarray(resized, dtype=np.uint8) > 14)
    mask = maximum_filter1d(mask, size=2 * dilate_y + 1, axis=0)
    return mask

def numeric_register_1d(x, x0, x1, original_width=1080, target_w=600):
    x_orig = (np.arange(target_w, dtype=np.float64) + .5) * original_width / target_w - .5
    valid = (x_orig >= x0) & (x_orig <= x1)
    t = np.clip((x_orig - x0) / (x1 - x0), 0, 1)
    out = np.interp(t * (len(x) - 1), np.arange(len(x)), x)
    return out, valid

_, GEOM_VALID = numeric_register_1d(np.zeros(N), X0, X1)

def make_perturbations(x, rng):
    p2p, sd, t = np.ptp(x), x.std(), np.arange(N) / FS
    names, fams, P = [], [], []
    for s in SCALES:
        names.append(f"scale_{s}"); fams.append("scale"); P.append(x * s)
    for a in DRIFTS:
        for ph in DRIFT_PHASES:
            names.append(f"drift_{a}_{ph:.3f}"); fams.append(f"drift_{a}")
            P.append(x + a * p2p * np.sin(2 * np.pi * DRIFT_HZ * t + ph))
    for a in NOISES:
        for k in range(NOISE_SEEDS):
            names.append(f"noise_{a}_{k}"); fams.append(f"noise_{a}")
            P.append(x + rng.normal(0, a * sd, N))
    P = np.stack(P)
    assert P.shape == (N_PERT, N)
    return names, fams, P

class Renderer:
    def __init__(self):
        self.fig = plt.figure(figsize=(15, 4), dpi=72)
        self.ax = self.fig.add_subplot(111)
        (self.line,) = self.ax.plot(np.arange(N), np.zeros(N))
        self.ax.axis("off")
    def base(self, x):
        self.ax.set_autoscaley_on(True)
        self.line.set_ydata(x)
        self.ax.relim(); self.ax.autoscale_view()
        self.fig.tight_layout()
        self.ylim = self.ax.get_ylim()
        return self._grab()
    def pert(self, x):
        self.line.set_ydata(x)
        self.ax.set_ylim(*self.ylim)
        return self._grab()
    def _grab(self):
        self.fig.canvas.draw()
        return np.asarray(self.fig.canvas.buffer_rgba())[..., :3].copy()

def to_2d(rgb):
    with tf.device("/CPU:0"):
        g = tf.image.rgb_to_grayscale(tf.cast(np.asarray(rgb), tf.float32))
        return tf.image.resize(g, [300, 600]).numpy()

def _chunked(key, fn, x):
    while True:
        bs = BS[key]
        try:
            dev = tf.device("/CPU:0") if FORCE_CPU[key] else contextlib.nullcontext()
            with dev:
                return np.concatenate([fn(x[i:i + bs]) for i in range(0, len(x), bs)])
        except tf.errors.OpError as e:
            if bs > 1:
                BS[key] = max(1, bs // 2)
                print(f"[{key}] {type(e).__name__}; retrying with batch {BS[key]}")
            elif not FORCE_CPU[key]:
                FORCE_CPU[key] = True
                print(f"[{key}] batch 1 failed on GPU; retrying on CPU")
            else:
                raise

def run_model(key, model, x):
    return _chunked(key, lambda xb: model(tf.convert_to_tensor(xb), training=False).numpy(), x)

_GM = {}
def gradcam(key, model, layer, x, k):
    """Raw ReLU'd Grad-CAM at native resolution (same target/layer/weights as the authors' scripts)."""
    if key not in _GM:
        _GM[key] = tf.keras.Model(model.inputs, [model.get_layer(layer).output, model.output])
    gm = _GM[key]
    def one(xb):
        with tf.GradientTape() as tape:
            conv, pred = gm(tf.convert_to_tensor(xb), training=False)
            loss = tf.reduce_sum(pred[:, k])
        g = tape.gradient(loss, conv)
        w = tf.reduce_mean(g, axis=tuple(range(1, len(conv.shape) - 1)), keepdims=True)
        return tf.nn.relu(tf.reduce_sum(w * conv, axis=-1)).numpy()
    return _chunked(key, one, x)

def _max_norm(c):                      
    m = float(np.max(c))
    return c / m if m > 0 else np.zeros_like(c, dtype=np.float32)

def _norm_or_nan(raw, valid):
    try:
        g = grayscale_norm(raw, valid)
    except ValueError:
        g = None
    if g is None:
        return np.full(W, np.nan, np.float32)
    g[~valid] = np.nan                 
    return g

def project_1d(cam_native, valid):
    c = _max_norm(np.asarray(cam_native, np.float64))
    full = np.interp(np.arange(N), np.linspace(0, N, len(c)), c)      
    reg, _ = numeric_register_1d(full, X0, X1)
    return _norm_or_nan(reg, valid)

def project_2d(cam_native, rgb, valid):
    c = _max_norm(np.asarray(cam_native, np.float64))
    r = tf.image.resize(c[..., None].astype(np.float32), [300, 600], method="bilinear").numpy()[..., 0]
    r = np.clip(r, 0.0, 1.0)
    mask = trace_mask_at_2d_grid(rgb, dilate_y=MASK_DILATE_Y)
    support = mask.sum(axis=0)
    raw = (r * mask).sum(axis=0) / (support + EPS)
    return _norm_or_nan(raw, valid), support

def stability(maps):                   
    mu, sd = maps.mean(0), maps.std(0)
    S = 1 - sd / (sd.max() + EPS)
    return mu, S, mu * S

def eval_record(x, rng, rend):
    names, fams, P = make_perturbations(x, rng)
    xs = np.concatenate([x[None], P])                      
    X1_ = ((xs - MEAN) / SCALE).astype(np.float32)[..., None]
    rgbs = np.stack([rend.base(x)] + [rend.pert(p) for p in P])       
    assert rgbs.shape[1:] == (288, 1080, 3), rgbs.shape
    I2 = to_2d(rgbs)
    I2n = (I2 / 255.0).astype(np.float32)

    pr1 = run_model("1d", m1, X1_)
    pr2 = run_model("2d", m2, I2n)
    k1, k2 = int(pr1[0].argmax()), int(pr2[0].argmax())
    keep1 = pr1.argmax(1)[1:] == k1
    keep2 = pr2.argmax(1)[1:] == k2

    cam1 = gradcam("1d", m1, L1, X1_, k1)
    cam2 = gradcam("2d", m2, L2, I2n, k2)

    mask0 = trace_mask_at_2d_grid(rgbs[0], dilate_y=MASK_DILATE_Y)
    valid = GEOM_VALID & (mask0.sum(axis=0) > 0)
    coverage = float(valid.sum() / max(1, GEOM_VALID.sum()))

    G1 = np.stack([project_1d(c, valid) for c in cam1])               
    G2 = np.stack([project_2d(c, rgbs[i], valid)[0] for i, c in enumerate(cam2)])

    lo, hi = rend.ylim
    clip = ((P < lo) | (P > hi)).mean(1)
    return dict(k1=k1, k2=k2, keep1=keep1, keep2=keep2, names=names, fams=np.array(fams),
                G1=G1, G2=G2, valid=valid, coverage=coverage, clip=clip,
                out1=pr1[0], out2=pr2[0])

def summarize(r, ev):
    k1, k2, keep1, keep2, valid = ev["k1"], ev["k2"], ev["keep1"], ev["keep2"], ev["valid"]
    G1, G2 = ev["G1"], ev["G2"]
    ok = lambda G: np.array([np.isfinite(g[valid]).all() and valid.any() for g in G])
    ok1, ok2 = ok(G1), ok(G2)                       
    acc1 = keep1 & ok1[1:]
    acc2 = keep2 & ok2[1:]
    joint = acc1 & acc2
    n1, n2, nj = int(acc1.sum()), int(acc2.sum()), int(joint.sum())

    row = dict(
        cohort_index=int(r.cohort_index), ecg_csv_filename=r.ecg_csv_filename,
        class_name=r.class_name, pred_1d=CLASSES[k1], pred_2d=CLASSES[k2],
        branch_disagree=bool(k1 != k2), n_perturb=N_PERT,
        n_kept_1d=int(keep1.sum()), n_kept_2d=int(keep2.sum()),
        n_degenerate_1d=int((~ok1[1:]).sum()), n_degenerate_2d=int((~ok2[1:]).sum()),
        n_accepted_1d=n1, n_accepted_2d=n2, n_joint_accepted=nj,
        prr_1d=float(keep1.mean()), prr_2d=float(keep2.mean()),
        base_degenerate_1d=bool(not ok1[0]), base_degenerate_2d=bool(not ok2[0]),
        base_coverage=ev["coverage"], bad_mask=bool(ev["coverage"] < COVERAGE_MIN),
        n_valid_cols=int(valid.sum()),
        clip_frac_mean=float(ev["clip"].mean()), clip_frac_max=float(ev["clip"].max()),
        S1_mean=np.nan, H1_mean=np.nan, S2_mean=np.nan, H2_mean=np.nan, R_mean=np.nan,
        status_1d="unevaluable", status_2d="unevaluable", status_joint="unevaluable")
    for fam in FAMILIES:
        idx = ev["fams"] == fam
        row[f"prr_1d_{fam}"] = float(keep1[idx].mean())
        row[f"prr_2d_{fam}"] = float(keep2[idx].mean())

    arr = {k: np.full(W, np.nan, np.float32) for k in
           ["mu1", "S1", "H1", "mu2", "S2", "H2", "H1j", "H2j", "R", "base1", "base2"]}
    arr["base1"], arr["base2"] = G1[0].astype(np.float32), G2[0].astype(np.float32)

    def put(keys, vals):
        for k, v in zip(keys, vals):
            arr[k][valid] = v

    if n1 >= MIN_ACCEPTED:
        mu, S, H = stability(G1[1:][acc1][:, valid])
        put(["mu1", "S1", "H1"], [mu, S, H])
        row.update(S1_mean=float(S.mean()), H1_mean=float(H.mean()), status_1d="evaluable")
    if n2 >= MIN_ACCEPTED:
        mu, S, H = stability(G2[1:][acc2][:, valid])
        put(["mu2", "S2", "H2"], [mu, S, H])
        row.update(S2_mean=float(S.mean()), H2_mean=float(H.mean()), status_2d="evaluable")
    if nj >= MIN_ACCEPTED:
        _, _, H1j = stability(G1[1:][joint][:, valid])
        _, _, H2j = stability(G2[1:][joint][:, valid])
        R = np.sqrt(H1j * H2j)
        put(["H1j", "H2j", "R"], [H1j, H2j, R])
        row.update(R_mean=float(R.mean()), status_joint="evaluable")

    return dict(row=row, arr=arr, valid=valid, keep1=keep1, keep2=keep2,
                deg1=~ok1[1:], deg2=~ok2[1:], names=ev["names"],
                out1=ev["out1"].astype(np.float32), out2=ev["out2"].astype(np.float32))

def save_outputs(records, out_dir, done):
    df = pd.DataFrame([rc["row"] for rc in records])
    df.to_csv(f"{out_dir}/noise_stability_records.csv", index=False)
    np.savez_compressed(
        f"{out_dir}/noise_stability_maps.npz",
        cohort_index=df["cohort_index"].to_numpy(dtype=np.int64),
        filename=np.array(df["ecg_csv_filename"].tolist(), dtype=str),
        pert_names=np.array(records[0]["names"]),
        valid_600=np.stack([rc["valid"] for rc in records]),
        keep_1d=np.stack([rc["keep1"] for rc in records]),
        keep_2d=np.stack([rc["keep2"] for rc in records]),
        degenerate_1d=np.stack([rc["deg1"] for rc in records]),
        degenerate_2d=np.stack([rc["deg2"] for rc in records]),
        out1_base=np.stack([rc["out1"] for rc in records]),
        out2_base=np.stack([rc["out2"] for rc in records]),
        **{k: np.stack([rc["arr"][k] for rc in records]) for k in records[0]["arr"]})
    with open(f"{out_dir}/run_metadata.json", "w") as f:
        json.dump(dict(config=CONFIG, n_records=len(df), done=done,
                       time=datetime.datetime.now().isoformat(timespec="seconds"),
                       tf=tf.__version__, batch_sizes=BS, force_cpu=FORCE_CPU,
                       scaler_n_fit=SCALER_N_FIT,
                       note="PROVISIONAL: scaler fitted on 9,592 records (intended 9,534); 2D rendering "
                            "not fully verified; perturbation magnitudes not validated"), f, indent=2)
    return df

def _atomic_pickle(obj, path):
    with open(path + ".tmp", "wb") as f:
        pickle.dump(obj, f)
    os.replace(path + ".tmp", path)

def run(rows, tag, resume=True):
    assert rows["ecg_csv_filename"].is_unique and rows["cohort_index"].is_unique
    out_dir = os.path.join(OUT, f"module2_{tag}")
    os.makedirs(out_dir, exist_ok=True)
    ckpt = os.path.join(out_dir, "checkpoint.pkl")
    records, done = [], set()
    if resume and os.path.exists(ckpt):
        with open(ckpt, "rb") as f:
            d = pickle.load(f)
        assert d["config"] == CONFIG, f"Config changed since checkpoint; use a new tag or delete {ckpt}"
        records = d["records"]
        done = {rc["row"]["cohort_index"] for rc in records}
        print(f"Resuming: {len(done)} records already done")
    rend, t0, todo = Renderer(), time.time(), 0
    try:
        for r in rows.itertuples(index=False):
            if int(r.cohort_index) in done:
                continue
            x = load_ecg(r.ecg_csv_filename)                   
            assert x.shape == (N,), x.shape
            ev = eval_record(x, np.random.default_rng(int(r.cohort_index)), rend)
            records.append(summarize(r, ev))
            todo += 1
            if todo % CKPT_EVERY == 0:
                _atomic_pickle(dict(config=CONFIG, records=records), ckpt)
                el = time.time() - t0
                print(f"{len(records)}/{len(rows)} | {el / todo:.1f}s/record | "
                      f"~{el / todo * (len(rows) - len(records)) / 60:.0f} min left", flush=True)
    finally:
        plt.close(rend.fig)
        if records:
            _atomic_pickle(dict(config=CONFIG, records=records), ckpt)
    df = save_outputs(records, out_dir, done=len(records))
    print("\n========== MODULE 2 v2 SUMMARY ==========")
    print("Output dir:", out_dir, "| records:", len(df))
    print("Evaluable 1D / 2D / joint:", (df.status_1d == "evaluable").sum(),
          (df.status_2d == "evaluable").sum(), (df.status_joint == "evaluable").sum())
    print("Mean PRR 1D / 2D:", round(df.prr_1d.mean(), 4), round(df.prr_2d.mean(), 4))
    print("Degenerate perturbed maps (total) 1D / 2D:", int(df.n_degenerate_1d.sum()), int(df.n_degenerate_2d.sum()))
    print("Degenerate BASE maps 1D / 2D:", int(df.base_degenerate_1d.sum()), int(df.base_degenerate_2d.sum()))
    print("Base mask coverage: min", round(df.base_coverage.min(), 3), "| bad_mask records:", int(df.bad_mask.sum()))
    print("Batch sizes in use:", BS, "| CPU fallback:", FORCE_CPU)
    print(df.groupby("class_name")[["prr_1d", "prr_2d", "n_joint_accepted", "S1_mean", "S2_mean", "R_mean"]]
          .mean().round(4))
    return df

def selftest():
    x = load_ecg(test.iloc[0]["ecg_csv_filename"])
    rend = Renderer()
    rgb = rend.base(x)
    img = (to_2d(rgb[None]) / 255.0).astype(np.float32)
    print("2D, 1 input :", run_model("2d", m2, img).round(3))
    print("2D, 31 inputs:", run_model("2d", m2, np.repeat(img, 31, 0)).shape)
    print("1D, 1 input :", run_model("1d", m1, ((x - MEAN) / SCALE).astype(np.float32)[None, :, None]).round(3))
    cov = (GEOM_VALID & (trace_mask_at_2d_grid(rgb).sum(0) > 0)).sum() / GEOM_VALID.sum()
    print("Trace-mask coverage on our rendered image (Module 1 requires >= 0.75):", round(float(cov), 3))
    plt.close(rend.fig)

print("Module 2 v2 definitions loaded | geometric valid columns:", int(GEOM_VALID.sum()), "of", W)


In [ ]:
# 8. Self-test: one 2D input, then 31 inputs (the size that used to crash), then one 1D input
selftest()


In [ ]:
# 10. Smoke test (2 records per class) -> gate -> full 1,054-record run (about an hour)
RUN_FULL = True
smoke = run(test.groupby("class_name", sort=False).head(2), tag="smoke8_v2")   
gate = ((smoke.status_joint == "evaluable").all() and smoke.base_coverage.min() >= COVERAGE_MIN
        and smoke.n_joint_accepted.min() >= 3)
print("\nSMOKE GATE:", "PASS" if gate else "FAIL - not starting the full run; send me the summary above")
if RUN_FULL and gate:
    results = run(test, tag="full_v2")


## Analyse the saved noise-stability run

In [ ]:
# 11. Analysis of the saved full v2 run (reads files only; no models, no GPU)
TAG = "full_v2"
D = os.path.join(OUT, f"module2_{TAG}")
df = pd.read_csv(f"{D}/noise_stability_records.csv")
z = np.load(f"{D}/noise_stability_maps.npz")

assert len(df) == len(test) and df.cohort_index.is_unique
assert set(df.cohort_index) == set(test.cohort_index)
assert (z["cohort_index"] == df.cohort_index.to_numpy()).all()
assert (z["filename"] == np.array(df.ecg_csv_filename.tolist())).all()
m = test.set_index("cohort_index").loc[df.cohort_index]
assert (m.ecg_csv_filename.to_numpy() == df.ecg_csv_filename.to_numpy()).all()
assert (m.class_name.to_numpy() == df.class_name.to_numpy()).all()
v = z["valid_600"]
assert np.isnan(z["R"][~v]).all()
print("integrity OK | rows:", len(df), "| records with any NaN R on valid columns:",
      int(np.isnan(np.where(v, z["R"], 0)).any(1).sum()))

print("\nEvaluable 1D / 2D / joint:", (df.status_1d == "evaluable").sum(),
      (df.status_2d == "evaluable").sum(), (df.status_joint == "evaluable").sum())
print("Degenerate perturbed maps 1D / 2D:", int(df.n_degenerate_1d.sum()), int(df.n_degenerate_2d.sum()))
print("Degenerate BASE maps 1D / 2D:", int(df.base_degenerate_1d.sum()), int(df.base_degenerate_2d.sum()))
print("Base mask coverage: min", round(df.base_coverage.min(), 3), "| bad_mask records (<0.75):", int(df.bad_mask.sum()),
      df[df.bad_mask].class_name.value_counts().to_dict())

fam = sorted(c for c in df.columns if c.startswith(("prr_1d_", "prr_2d_")))
print("\nPRR by family (overall):")
print(df[fam].mean().round(4).to_string())
for b in ["1d", "2d"]:
    print(f"\nPRR by family x class ({b.upper()}):")
    print(df.groupby("class_name")[[c for c in fam if c.startswith(f"prr_{b}_")]].mean().round(3).to_string())

for c in ["prr_1d", "prr_2d"]:
    x = df[c]
    print(f"\n{c}: min={x.min():.3f} p5={x.quantile(.05):.3f} median={x.median():.3f} | "
          f"<0.9: {(x < .9).sum()}  <0.5: {(x < .5).sum()}  ==1.0: {(x == 1).sum()}")
print("n_joint_accepted: min", df.n_joint_accepted.min(), "| records with <10:", (df.n_joint_accepted < 10).sum())

print("\nBase accuracy 1D / 2D:", round((df.pred_1d == df.class_name).mean(), 4),
      round((df.pred_2d == df.class_name).mean(), 4))
print("Branch disagreement:", round(df.branch_disagree.mean(), 4))
print(df.groupby("class_name").branch_disagree.mean().round(4).to_string())

print("\nCorrelation S1_mean vs S2_mean:", round(df.S1_mean.corr(df.S2_mean), 3))
print(df[["S1_mean", "S2_mean", "H1_mean", "H2_mean", "R_mean"]].describe().round(4).to_string())
print("\nBy class:")
print(df.groupby("class_name")[["prr_1d", "prr_2d", "n_joint_accepted", "S1_mean", "S2_mean", "R_mean"]].mean().round(4))
print("\nStability by whether the branches agree on the base prediction:")
print(df.groupby("branch_disagree")[["S1_mean", "S2_mean", "R_mean"]].mean().round(4).to_string())

g = np.linspace(0, 10, 600)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6), sharex=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    for cls in sorted(df.class_name.unique()):
        i = (df.class_name == cls).to_numpy()
        for a, k, t in zip(ax, ["R", "S1", "S2"], ["R(u)", "S_1D(u)", "S_2D(u)"]):
            a.plot(g, np.nanmean(z[k][i], 0), label=cls); a.set_title(t + " - class mean"); a.set_xlabel("time (s)")
ax[0].legend(); plt.tight_layout(); plt.savefig(f"{D}/class_mean_maps.png", dpi=120); plt.show()


## Generate noise-stability figures

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from PIL import Image
import gc

TAG = "full_v2"
D = f"/kaggle/working/m2_out/module2_{TAG}"
df = pd.read_csv(f"{D}/noise_stability_records.csv")
z = np.load(f"{D}/noise_stability_maps.npz")
ECG_ROOT = "/kaggle/input/datasets/aihash4/ecgdata/ECGData"
FIG_DIR = "/kaggle/working/m2_out/figures_m2"
os.makedirs(FIG_DIR, exist_ok=True)

def load_ecg(fn):
    return pd.read_csv(os.path.join(ECG_ROOT, fn), usecols=["II"])["II"].to_numpy(dtype=np.float64)

BG = "#FFFBF7"       
INK = "#2D3047"      
MUTE = "#767A8C"     
EDGE = "#E8E0DA"     
CLS = {"AFib": "#D06B9A", "GSVT": "#4A93B5", "SB": "#A98568", "SR": "#A47A8A"}
CL = ["AFib", "GSVT", "SB", "SR"]

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": "white", 
    "savefig.facecolor": BG, "axes.edgecolor": EDGE,
    "axes.labelcolor": MUTE, "xtick.color": MUTE, "ytick.color": MUTE,
    "text.color": INK, "font.size": 11, "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False
})

def save_fig(fig, name):
    path = f"{FIG_DIR}/{name}.png"
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {name}.png")

def draw_card(fig, x, y, w, h):
    fig.add_artist(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=0.012", 
                                  transform=fig.transFigure, fc="white", ec=EDGE, lw=1.2, zorder=0))

def make_dashboard():
    fig = plt.figure(figsize=(16, 9))
    fig.text(0.04, 0.92, "DAN / MMGradCAM", color=CLS["AFib"], weight="bold", fontsize=14)
    fig.text(0.04, 0.88, "02 | Noise Stability Overview", fontsize=28, weight="bold")
    fig.text(0.04, 0.84, f"{len(df):,} held-out ECG records  ·  30 perturbations each  ·  Module 2 focus", fontsize=13, color=MUTE)

    kpis = [
        ("RECORDS", f"{len(df):,}", CLS["AFib"]),
        ("MEDIAN S (1D)", f"{df.S1_mean.median():.3f}", CLS["GSVT"]),
        ("MEDIAN S (2D)", f"{df.S2_mean.median():.3f}", CLS["SB"]),
        ("MEDIAN R", f"{df.R_mean.median():.3f}", CLS["SR"]),
        ("MEAN PRR", f"{df.prr_1d.mean():.2f} / {df.prr_2d.mean():.2f}", INK)
    ]
    for k, (title, val, col) in enumerate(kpis):
        x = 0.04 + k * 0.19
        draw_card(fig, x, 0.70, 0.17, 0.11)
        fig.text(x + 0.02, 0.76, title, color=MUTE, weight="bold", fontsize=9)
        fig.text(x + 0.02, 0.72, val, color=col, weight="bold", fontsize=22)

    ax1 = fig.add_axes([0.06, 0.12, 0.40, 0.48])
    fams = ['scale', 'drift_0.05', 'drift_0.1', 'noise_0.05', 'noise_0.1']
    labels = ['Scale', 'Drift 5%', 'Drift 10%', 'Noise 5%', 'Noise 10%']
    x = np.arange(len(fams))
    ax1.bar(x - 0.2, [df[f'prr_1d_{f}'].mean() for f in fams], 0.4, label='1D', color=CLS["GSVT"])
    ax1.bar(x + 0.2, [df[f'prr_2d_{f}'].mean() for f in fams], 0.4, label='2D', color=CLS["AFib"])
    ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=15)
    ax1.set_ylim(0.85, 1.0); ax1.legend(frameon=False)
    ax1.set_title("Prediction Retention (PRR) by Perturbation", loc='left', fontweight='bold')
    ax1.grid(axis='y', color=EDGE)

    ax2 = fig.add_axes([0.56, 0.12, 0.38, 0.48])
    for j, c in enumerate(CL):
        bp = ax2.boxplot([df[df.class_name == c]['S1_mean'], df[df.class_name == c]['S2_mean']], 
                         positions=[j-0.15, j+0.15], widths=0.25, patch_artist=True, showfliers=False)
        bp['boxes'][0].set(facecolor=CLS["GSVT"], alpha=0.8)
        bp['boxes'][1].set(facecolor=CLS["AFib"], alpha=0.8)
    ax2.set_xticks(range(4)); ax2.set_xticklabels(CL)
    ax2.set_title("Explanation Stability (S) by Rhythm", loc='left', fontweight='bold')
    ax2.set_ylabel("Stability Score (S)")
    ax2.grid(axis='y', color=EDGE)
    
    fig.text(0.04, 0.02, "Solid boxes = 1D Model  ·  Pale boxes = 2D Model  ·  Stability is not a faithfulness claim", color=MUTE, fontsize=9)
    save_fig(fig, "01_dashboard_overview")

def make_distributions():
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    ax = axes.ravel()
    
    ax[0].hist([df.prr_1d, df.prr_2d], bins=np.linspace(0.1, 1.0, 25), color=[CLS["GSVT"], CLS["AFib"]], label=["1D", "2D"])
    ax[0].set_title("Prediction Retention Rate (PRR)")
    ax[0].legend(frameon=False); ax[0].set_yscale('log')
    
    ax[1].hist([df.S1_mean, df.S2_mean], bins=30, color=[CLS["GSVT"], CLS["AFib"]], label=["S1 (1D)", "S2 (2D)"])
    ax[1].set_title("Mean Stability per Record")
    ax[1].legend(frameon=False)
    
    ax[2].hist(df.R_mean, bins=30, color=CLS["SR"])
    ax[2].set_title("Mean Reliability R(u) per Record")
    
    ax[3].hist(df.n_joint_accepted, bins=np.arange(0, 32, 1), color=CLS["SB"])
    ax[3].set_title("Jointly Accepted Perturbations (out of 30)")
    
    for a in ax: a.grid(axis='y', color=EDGE)
    fig.tight_layout()
    save_fig(fig, "02_distributions")

def make_scatter():
    fig, ax = plt.subplots(figsize=(8, 6))
    for c in CL:
        d = df[df.class_name == c]
        ax.scatter(d.S1_mean, d.S2_mean, s=20, alpha=0.6, color=CLS[c], label=c, edgecolors='none')
    ax.set_xlabel("1D Stability (S1)")
    ax.set_ylabel("2D Stability (S2)")
    ax.set_title(f"Stability Comparison: 1D vs 2D (r = {df.S1_mean.corr(df.S2_mean):.2f})", fontweight='bold')
    ax.legend(frameon=False)
    ax.grid(color=EDGE)
    save_fig(fig, "03_scatter_S1_vs_S2")

def make_class_curves():
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True)
    W_, FS_ = 600, 500.0
    TS = np.clip(((np.arange(W_) + .5) * 1080 / W_ - .5 - 58) / (1021 - 58), 0, 1) * (5000 - 1)
    TSEC = TS / FS_
    v = np.isfinite(z["R"]).any(0)
    
    for c in CL:
        i = (df.class_name == c).to_numpy()
        for a, k, t in zip(ax, ["R", "S1", "S2"], ["R(u)", "S 1D(u)", "S 2D(u)"]):
            m = np.nanmean(z[k][i], 0)
            a.plot(TSEC[v], m[v], color=CLS[c], label=c)
            a.set_title(f"{t}  (Class Mean)")
            a.set_xlabel("Time (s)")
            a.grid(color=EDGE)
    ax[0].legend(frameon=False)
    fig.tight_layout()
    save_fig(fig, "04_class_mean_curves")

def make_heatmaps():
    o = df.assign(_i=np.arange(len(df))).sort_values(["class_name", "R_mean"])
    idx = o._i.to_numpy()
    v = np.isfinite(z["R"]).any(0)
    
    fig, ax = plt.subplots(1, 3, figsize=(17, 8), sharey=True)
    for a, k, t in zip(ax, ["R", "S1", "S2"], ["R(u)", "S 1D(u)", "S 2D(u)"]):
        M = z[k][idx][:, v]
        hi = np.nanpercentile(M, 99)
        im = a.imshow(M, aspect="auto", cmap="magma", vmin=0, vmax=hi, interpolation="nearest")
        a.set_title(f"{t} (Sorted by class & R)")
        a.set_xlabel("Time Grid (600 cols)")
        fig.colorbar(im, ax=a, fraction=.04)
        
    cum = np.cumsum([0] + [int((o.class_name == c).sum()) for c in CL])
    for a in ax:
        for e in cum[1:-1]: a.axhline(e, color="white", lw=1.5, linestyle="--")
    ax[0].set_yticks((cum[:-1] + cum[1:]) / 2)
    ax[0].set_yticklabels(CL)
    fig.tight_layout()
    save_fig(fig, "05_record_by_time_heatmaps")

def make_panels():
    ok = df[(df.pred_1d == df.class_name) & (df.pred_2d == df.class_name)]
    t_axis = np.linspace(0, 10, 600) 
    
    for c in ["AFib", "SR"]:
        d = ok[ok.class_name == c].sort_values("R_mean")
        for label, r in [("Most Stable", d.iloc[-1]), ("Median", d.iloc[len(d)//2]), ("Least Stable", d.iloc[0])]:
            i = int(np.where(df.cohort_index.to_numpy() == r.cohort_index)[0][0])
            x = load_ecg(r.ecg_csv_filename)
            
            fig = plt.figure(figsize=(14, 7))
            gs = fig.add_gridspec(4, 1, height_ratios=[2, 1, 1, 1], hspace=0.4)

            ax0 = fig.add_subplot(gs[0])
            ax0.plot(np.arange(5000)/500, x, color=INK, lw=1)
            ax0.set_xlim(0, 10)
            ax0.set_title(f"{c} · {label} · Cohort {int(r.cohort_index)} · R={r.R_mean:.3f}", loc='left', fontweight='bold')
            ax0.set_xticks([]); ax0.set_yticks([])

            ax1 = fig.add_subplot(gs[1])
            ax1.imshow(z['base2'][i][np.newaxis, :], aspect='auto', cmap='jet', interpolation='bilinear', extent=[0, 10, 0, 1])
            ax1.set_ylabel("2D CAM", rotation=0, ha='right', va='center')
            ax1.set_xticks([]); ax1.set_yticks([])

            ax2 = fig.add_subplot(gs[2])
            ax2.plot(t_axis, z['S1'][i], color=CLS["GSVT"], label="S1 (1D)")
            ax2.plot(t_axis, z['S2'][i], color=CLS["AFib"], label="S2 (2D)")
            ax2.set_xlim(0, 10)
            ax2.set_ylabel("Stability S", rotation=0, ha='right', va='center')
            ax2.legend(frameon=False, ncol=2, loc="upper right")
            ax2.set_xticks([])

            ax3 = fig.add_subplot(gs[3])
            ax3.fill_between(t_axis, z['R'][i], color=CLS["SR"], alpha=0.6)
            ax3.plot(t_axis, z['R'][i], color=CLS["SR"])
            ax3.set_xlim(0, 10)
            ax3.set_ylabel("R(u)", rotation=0, ha='right', va='center')
            ax3.set_xlabel("Time (seconds)")
            
            save_fig(fig, f"06_panel_{c}_{label.replace(' ', '_')}")
            gc.collect()

def make_gallery():
    classes_to_plot = ["AFib", "GSVT", "SB"]
    selected_indices = {}
    for c in classes_to_plot:
        class_df = df[df.class_name == c].sort_values("R_mean")
        selected_indices[c] = class_df.index[len(class_df) // 2]

    fig = plt.figure(figsize=(16, 10))
    fig.text(0.04, 0.94, "Anchor-style ECG alignment gallery", fontsize=26, weight="bold", color=INK)
    fig.text(0.04, 0.90, "Module 2: Noise Stability & Reliability-weighted attribution R(u)", fontsize=13, color=MUTE)

    for i, cls in enumerate(classes_to_plot):
        idx = selected_indices[cls]
        row = df.iloc[idx]
        x = load_ecg(row.ecg_csv_filename)
        
        base2 = z['base2'][idx]
        R_map = z['R'][idx]
        
        y0 = 0.68 - i * 0.23
        draw_card(fig, 0.04, y0, 0.92, 0.20)
        
        fig.text(0.06, y0 + 0.16, cls, color="white", weight="bold", fontsize=12, 
                 bbox=dict(boxstyle="round,pad=0.4", fc=CLS[cls], ec="none"))
        fig.text(0.14, y0 + 0.16, "Paired ECG  ·  1D markers + 2D CAM", fontsize=11, color=INK)
        
        ax = fig.add_axes([0.10, y0 + 0.03, 0.82, 0.11])
        ax.plot(np.linspace(0, 10, 5000), x, color=INK, linewidth=0.8, zorder=3)
        ax.set_xlim(0, 10)
        ax.set_ylim(x.min(), x.max())
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values(): spine.set_visible(False)
        
        ax.imshow(np.ma.masked_where(base2 < 0.15, base2)[np.newaxis, :], 
                  aspect='auto', cmap='Spectral_r', alpha=0.6, zorder=2,
                  extent=[0, 10, x.min(), x.max()], interpolation='bilinear')
        
        base1_5000 = np.interp(np.linspace(0, 10, 5000), np.linspace(0, 10, 600), z['base1'][idx])
        thr = np.percentile(base1_5000, 85)
        marker_idx = np.where(base1_5000 > thr)[0]
        ax.scatter(np.linspace(0, 10, 5000)[marker_idx], x[marker_idx], 
                   color="#E0182D", s=8, zorder=4, edgecolors='none')
        
        ax_R = fig.add_axes([0.10, y0 + 0.03, 0.82, 0.02])
        ax_R.imshow(R_map[np.newaxis, :], aspect='auto', cmap='magma', 
                    extent=[0, 10, 0, 1], vmin=0, vmax=np.nanpercentile(R_map, 99), interpolation='bilinear')
        ax_R.set_xticks([]); ax_R.set_yticks([])
        for spine in ax_R.spines.values(): spine.set_visible(False)
        fig.text(0.06, y0 + 0.03, "R(u)", color=MUTE, fontsize=10, weight="bold")

    fig.text(0.04, 0.02, "Paper-like visual composition; provisional temporal co-registration; not a quantitative agreement validation.", color=MUTE, fontsize=10)
    save_fig(fig, "07_anchor_gallery")

print("Generating all Module 2 figures...")
make_dashboard()
make_distributions()
make_scatter()
make_class_curves()
make_heatmaps()
make_panels()
make_gallery()
print("All figures generated successfully in /kaggle/working/m2_out/figures_m2/")


In [ ]:
import os
from IPython.display import Image, display

FIG_DIR = "/kaggle/working/m2_out/figures_m2"

files_to_display = [
    "01_dashboard_overview.png",
    "02_distributions.png",
    "03_scatter_S1_vs_S2.png",
    "04_class_mean_curves.png",
    "05_record_by_time_heatmaps.png",
    
    "06_panel_AFib_Most_Stable.png",
    "06_panel_AFib_Median.png",
    "06_panel_AFib_Least_Stable.png",
    "06_panel_SR_Most_Stable.png",
    "06_panel_SR_Median.png",
    "06_panel_SR_Least_Stable.png",
    
    "07_anchor_gallery.png"
]

for filename in files_to_display:
    filepath = os.path.join(FIG_DIR, filename)
    if os.path.exists(filepath):
        print(f"\n{'='*60}\nDisplaying: {filename}\n{'='*60}")
        
        display(Image(filename=filepath, width=850))
    else:
        print(f"Missing (did not generate): {filename}")


## Faithfulness and validation analyses

In [ ]:
import os, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import bootstrap

OUT_DIR = "/kaggle/working/m2_out/faithfulness_corrected_v1"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 20261115
N_PER_CLASS = 25
BLOCK = 50                 
RANDOM_REPEATS = 5
FRACTIONS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
BOOTSTRAPS = 2000

X0, X1, ORIGINAL_WIDTH, GRID_WIDTH = 58, 1021, 1080, 600
rng = np.random.default_rng(SEED)

required = [
    "cohort_index", "out1_base", "out2_base",
    "R", "H1", "H2", "base1", "base2"
]
missing = [k for k in required if k not in z.files]
assert not missing, f"Missing saved arrays: {missing}"

assert len(df) == len(z["cohort_index"])
assert np.array_equal(
    df["cohort_index"].to_numpy(),
    z["cohort_index"]
), "DataFrame and saved map order do not match."

assert N == 5000
assert MEAN.shape == (N,), f"Unexpected MEAN shape: {MEAN.shape}"
assert SCALE.shape == (N,), f"Unexpected SCALE shape: {SCALE.shape}"

eligible = df[
    (df.pred_1d == df.class_name) &
    (df.pred_2d == df.class_name)
].copy()

selected_parts = []
for cls in CLASSES:
    group = eligible[eligible.class_name == cls]
    assert len(group) >= N_PER_CLASS, (
        f"Only {len(group)} eligible records for {cls}; "
        f"cannot select {N_PER_CLASS}."
    )
    selected_parts.append(
        group.sample(N_PER_CLASS, random_state=SEED)
    )

sample = (
    pd.concat(selected_parts)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

print("Selected records:", len(sample))
print(sample.class_name.value_counts().to_dict())

z_index = {
    int(c): i for i, c in enumerate(z["cohort_index"])
}

def map_to_signal(attribution):
    a = np.asarray(attribution, dtype=np.float64)
    valid = np.isfinite(a)

    assert valid.any(), "Attribution map contains no valid values."

    sample_pixels = (
        X0 + np.linspace(0, 1, N) * (X1 - X0)
    )
    sample_columns = (
        (sample_pixels + 0.5) * GRID_WIDTH / ORIGINAL_WIDTH - 0.5
    )

    return np.interp(
        sample_columns,
        np.flatnonzero(valid),
        a[valid]
    )

def choose_top_blocks(scores, n_blocks, block_size):
    """Greedily select high-scoring non-overlapping contiguous blocks."""
    scores = np.asarray(scores)
    candidates = []

    stride = 5
    for start in range(0, len(scores) - block_size + 1, stride):
        score = float(scores[start:start + block_size].mean())
        candidates.append((score, start))

    candidates.sort(reverse=True)

    occupied = np.zeros(len(scores), dtype=bool)
    selected = []

    for _, start in candidates:
        end = start + block_size
        if not occupied[start:end].any():
            selected.append(start)
            occupied[start:end] = True
            if len(selected) == n_blocks:
                break

    assert len(selected) == n_blocks, "Could not select enough non-overlapping blocks."
    return selected

def choose_random_blocks(n_blocks, block_size, seed):
    """Random non-overlapping blocks, matched in number and length."""
    local_rng = np.random.default_rng(seed)
    starts = np.arange(0, N - block_size + 1)
    local_rng.shuffle(starts)

    occupied = np.zeros(N, dtype=bool)
    selected = []

    for start in starts:
        end = start + block_size
        if not occupied[start:end].any():
            selected.append(int(start))
            occupied[start:end] = True
            if len(selected) == n_blocks:
                break

    assert len(selected) == n_blocks
    return selected

def occlude_blocks(x, starts, block_size):
    """
    Replace each selected block by linear interpolation between
    its neighbouring ECG samples, rather than setting samples to zero.
    """
    out = np.asarray(x, dtype=np.float64).copy()

    for start in starts:
        end = start + block_size

        left = out[start - 1] if start > 0 else out[end]
        right = out[end] if end < len(out) else left

        out[start:end] = np.linspace(
            left, right, block_size + 2
        )[1:-1]

    return out

def predict_1d(x):
    inp = ((x - MEAN) / SCALE).astype(np.float32)[None, :, None]
    return run_model("1d", m1, inp)[0]

def predict_2d(x, renderer):
    rgb = renderer.pert(x)
    image = (to_2d(rgb[None]) / 255.0).astype(np.float32)
    return run_model("2d", m2, image)[0]

METHODS = {
    "1d": ["R", "H1", "base1"],
    "2d": ["R", "H2", "base2"]
}

records = []
renderer = Renderer()

print("\nStarting corrected deletion experiment...")

try:
    for record_no, row in sample.iterrows():

        cohort = int(row.cohort_index)
        zi = z_index[cohort]
        x = load_ecg(row.ecg_csv_filename)

        assert x.shape == (N,), (
            f"Unexpected ECG length for cohort {cohort}: {x.shape}"
        )

        target = CLASSES.index(row.class_name)

        base1 = predict_1d(x)

        rgb0 = renderer.base(x)
        image0 = (to_2d(rgb0[None]) / 255.0).astype(np.float32)
        base2 = run_model("2d", m2, image0)[0]

        saved1 = z["out1_base"][zi]
        saved2 = z["out2_base"][zi]

        assert np.allclose(base1, saved1, atol=1e-5), (
            f"1D baseline mismatch: cohort {cohort}"
        )
        assert np.allclose(base2, saved2, atol=1e-5), (
            f"2D baseline mismatch: cohort {cohort}"
        )

        maps = {
            name: map_to_signal(z[name][zi])
            for name in ["R", "H1", "H2", "base1", "base2"]
        }

        assert all(np.isfinite(v).all() for v in maps.values())

        for branch, methods in METHODS.items():

            original_prob = (
                base1[target] if branch == "1d" else base2[target]
            )

            for method in methods:
                scores = maps[method]

                for fraction in FRACTIONS:
                    n_samples = int(round(N * fraction))
                    n_blocks = max(1, int(round(n_samples / BLOCK)))

                    actual_fraction = n_blocks * BLOCK / N

                    top_starts = choose_top_blocks(
                        scores, n_blocks, BLOCK
                    )
                    x_top = occlude_blocks(x, top_starts, BLOCK)

                    if branch == "1d":
                        top_prob = predict_1d(x_top)[target]
                    else:
                        top_prob = predict_2d(x_top, renderer)[target]

                    records.append({
                        "cohort_index": cohort,
                        "class_name": row.class_name,
                        "branch": branch,
                        "method": method,
                        "control": "attribution",
                        "requested_fraction": fraction,
                        "actual_fraction": actual_fraction,
                        "deleted_samples": n_blocks * BLOCK,
                        "baseline_probability": original_prob,
                        "perturbed_probability": float(top_prob),
                        "probability_drop": float(original_prob - top_prob)
                    })

                    for rep in range(RANDOM_REPEATS):
                        random_starts = choose_random_blocks(
                            n_blocks, BLOCK,
                            seed=SEED + cohort * 1000 +
                                 int(fraction * 10000) + rep
                        )

                        x_random = occlude_blocks(
                            x, random_starts, BLOCK
                        )

                        if branch == "1d":
                            random_prob = predict_1d(x_random)[target]
                        else:
                            random_prob = predict_2d(
                                x_random, renderer
                            )[target]

                        records.append({
                            "cohort_index": cohort,
                            "class_name": row.class_name,
                            "branch": branch,
                            "method": method,
                            "control": "random",
                            "requested_fraction": fraction,
                            "actual_fraction": actual_fraction,
                            "deleted_samples": n_blocks * BLOCK,
                            "baseline_probability": original_prob,
                            "perturbed_probability": float(random_prob),
                            "probability_drop": float(
                                original_prob - random_prob
                            )
                        })

        if (record_no + 1) % 10 == 0:
            print(
                f"Processed {record_no + 1}/{len(sample)} records",
                flush=True
            )
            gc.collect()

finally:
    plt.close(renderer.fig)
    gc.collect()

results_df = pd.DataFrame(records)

raw_path = os.path.join(OUT_DIR, "deletion_raw_results.csv")
results_df.to_csv(raw_path, index=False)

print("\nRaw results saved:", raw_path)

def bootstrap_mean_ci(values, seed):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 2:
        return np.nan, np.nan, np.nan

    result = bootstrap(
        (values,),
        np.mean,
        confidence_level=0.95,
        n_resamples=BOOTSTRAPS,
        method="percentile",
        random_state=np.random.default_rng(seed)
    )

    return (
        float(values.mean()),
        float(result.confidence_interval.low),
        float(result.confidence_interval.high)
    )

summary_rows = []

for (branch, method, fraction), group in results_df.groupby(
    ["branch", "method", "requested_fraction"]
):

    attr = (
        group[group.control == "attribution"]
        .set_index("cohort_index")["probability_drop"]
    )

    random = (
        group[group.control == "random"]
        .groupby("cohort_index")["probability_drop"]
        .mean()
    )

    paired = pd.concat(
        [attr.rename("attribution"), random.rename("random")],
        axis=1
    ).dropna()

    mean_a, lo_a, hi_a = bootstrap_mean_ci(
        paired.attribution, SEED
    )
    mean_r, lo_r, hi_r = bootstrap_mean_ci(
        paired.random, SEED + 1
    )
    mean_diff, lo_diff, hi_diff = bootstrap_mean_ci(
        paired.attribution - paired.random, SEED + 2
    )

    summary_rows.append({
        "branch": branch,
        "method": method,
        "fraction": fraction,
        "n_records": len(paired),
        "attribution_mean_drop": mean_a,
        "attribution_ci_low": lo_a,
        "attribution_ci_high": hi_a,
        "random_mean_drop": mean_r,
        "random_ci_low": lo_r,
        "random_ci_high": hi_r,
        "paired_difference": mean_diff,
        "difference_ci_low": lo_diff,
        "difference_ci_high": hi_diff
    })

summary = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUT_DIR, "deletion_summary_bootstrap.csv")
summary.to_csv(summary_path, index=False)

print("\nSUMMARY")
display(summary.round(4))

for branch in ["1d", "2d"]:
    fig, ax = plt.subplots(figsize=(9, 6))

    branch_summary = summary[summary.branch == branch]

    for method in METHODS[branch]:
        sub = branch_summary[branch_summary.method == method].sort_values(
            "fraction"
        )

        ax.plot(
            sub.fraction * 100,
            sub.attribution_mean_drop,
            marker="o",
            label=f"{method} — attribution"
        )

        ax.plot(
            sub.fraction * 100,
            sub.random_mean_drop,
            marker="s",
            linestyle="--",
            label=f"{method} — random"
        )

    ax.axhline(0, linewidth=0.8)
    ax.set_xlabel("ECG signal deleted (%)")
    ax.set_ylabel("Drop in true-class probability")
    ax.set_title(f"Corrected Deletion Faithfulness — {branch.upper()}")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
    fig.tight_layout()

    plot_path = os.path.join(
        OUT_DIR, f"deletion_faithfulness_{branch}.png"
    )
    fig.savefig(plot_path, dpi=200)
    plt.show()
    plt.close(fig)

print("\nCompleted.")
print("Output directory:", OUT_DIR)
print("Raw results:", raw_path)
print("Summary:", summary_path)


In [ ]:
summary = pd.read_csv(
    "/kaggle/working/m2_out/faithfulness_corrected_v1/deletion_summary_bootstrap.csv"
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

display(summary.round(4))


## Mapping audit

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AUDIT_DIR = "/kaggle/working/m2_out/mapping_audit_v1"
os.makedirs(AUDIT_DIR, exist_ok=True)

N = 5000
W = 600
X0, X1 = 58, 1021
ORIGINAL_WIDTH = 1080

cohorts = np.asarray(z["cohort_index"])
assert len(df) == len(cohorts)
assert np.array_equal(df["cohort_index"].to_numpy(), cohorts)

MAPS = ["R", "H1", "H2", "base1", "base2"]

for name in MAPS:
    assert name in z.files, f"Missing map: {name}"
    assert z[name].shape == (len(df), W), (
        f"{name} has unexpected shape {z[name].shape}"
    )

col_centres = (
    (np.arange(W) + 0.5) * ORIGINAL_WIDTH / W - 0.5
)
geom_valid = (col_centres >= X0) & (col_centres <= X1)

assert np.flatnonzero(geom_valid)[0] == 32
assert np.flatnonzero(geom_valid)[-1] == 567

sample_pixels = np.linspace(X0, X1, N)
sample_columns = (
    (sample_pixels + 0.5) * W / ORIGINAL_WIDTH - 0.5
)

recovered_pixels = (
    (sample_columns + 0.5) * ORIGINAL_WIDTH / W - 0.5
)
geometry_error = np.max(np.abs(recovered_pixels - sample_pixels))
assert geometry_error < 1e-9

def map_to_ecg(map_600):
    """Interpolate only from the geometrically valid ECG columns."""
    a = np.asarray(map_600, dtype=np.float64)
    valid = geom_valid & np.isfinite(a)

    assert valid.sum() > 0, "No valid attribution columns."

    return np.interp(
        sample_columns,
        np.arange(W)[valid],
        a[valid]
    )

audit_rows = []

for name in MAPS:
    arr = np.asarray(z[name], dtype=np.float64)

    outside_mass = []
    top_outside = []
    finite_coverage = []

    for a in arr:
        finite = np.isfinite(a)
        positive = np.where(finite, np.maximum(a, 0), 0)

        total = positive.sum()
        outside = positive[~geom_valid].sum()

        outside_mass.append(outside / total if total > 0 else np.nan)

        valid_scores = np.where(finite, a, -np.inf)
        if np.isfinite(valid_scores).any():
            threshold = np.quantile(
                valid_scores[np.isfinite(valid_scores)], 0.90
            )
            top = valid_scores >= threshold
            top_outside.append(
                (top & ~geom_valid).sum() / max(1, top.sum())
            )
        else:
            top_outside.append(np.nan)

        finite_coverage.append(
            (finite & geom_valid).sum() / geom_valid.sum()
        )

    audit_rows.append({
        "map": name,
        "mean_mass_outside_geometry": np.nanmean(outside_mass),
        "mean_top10_percent_outside_geometry": np.nanmean(top_outside),
        "mean_finite_coverage_inside_geometry": np.nanmean(finite_coverage)
    })

audit = pd.DataFrame(audit_rows)
audit.to_csv(
    os.path.join(AUDIT_DIR, "mapping_geometry_audit.csv"),
    index=False
)

print("Geometric support:", np.flatnonzero(geom_valid)[0],
      "to", np.flatnonzero(geom_valid)[-1])
print("Maximum coordinate round-trip error:", geometry_error)
display(audit.round(5))

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

for ax, cls in zip(axes, CLASSES):
    candidates = df.index[df.class_name == cls]
    idx = int(candidates[0])
    row = df.iloc[idx]

    x = load_ecg(row.ecg_csv_filename)
    assert len(x) == N

    x_display = (x - np.mean(x)) / (np.std(x) + 1e-8)
    t = np.arange(N) / 500.0

    ax.plot(t, x_display, color="black", alpha=0.55,
            linewidth=0.7, label="ECG")

    for name in ["R", "base1", "base2"]:
        signal_map = map_to_ecg(z[name][idx])
        signal_map = np.nan_to_num(signal_map, nan=0.0)
        signal_map = signal_map / (signal_map.max() + 1e-8)

        ax.plot(
            t, signal_map,
            linewidth=1.2,
            label=name
        )

    ax.set_title(f"{cls} — cohort {row.cohort_index}")
    ax.set_ylabel("Normalised value")
    ax.legend(loc="upper right", ncol=4)
    ax.grid(alpha=0.2)

axes[-1].set_xlabel("ECG time (seconds)")
fig.tight_layout()

fig.savefig(
    os.path.join(AUDIT_DIR, "mapping_overlay_by_class.png"),
    dpi=200
)
plt.show()
plt.close(fig)

print("\nMapping audit saved to:", AUDIT_DIR)


## Deletion sensitivity analysis

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt

SENS_DIR = "/kaggle/working/m2_out/deletion_sensitivity_v1"
os.makedirs(SENS_DIR, exist_ok=True)

SEED = 20261116
N_PER_CLASS = 10
BLOCK_SIZES = [25, 50, 100]
FRACTIONS = [0.10, 0.20, 0.30]
RANDOM_REPEATS = 5
BOOTSTRAPS = 2000

eligible = df[
    (df.pred_1d == df.class_name) &
    (df.pred_2d == df.class_name)
].copy()

parts = []
for cls in CLASSES:
    group = eligible[eligible.class_name == cls]
    assert len(group) >= N_PER_CLASS, f"Not enough records for {cls}"
    parts.append(group.sample(N_PER_CLASS, random_state=SEED))

sample = (
    pd.concat(parts)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

cohort_to_zi = {
    int(c): i for i, c in enumerate(z["cohort_index"])
}

col_centres = (
    (np.arange(600) + 0.5) * 1080 / 600 - 0.5
)
geom_valid = (col_centres >= 58) & (col_centres <= 1021)

sample_pixels = np.linspace(58, 1021, N)
sample_columns = (
    (sample_pixels + 0.5) * 600 / 1080 - 0.5
)

def map_to_ecg(map_600):
    a = np.asarray(map_600, dtype=float)
    valid = geom_valid & np.isfinite(a)
    assert valid.any(), "Map has no valid ECG support."
    return np.interp(
        sample_columns,
        np.arange(600)[valid],
        a[valid]
    )

def select_top_blocks(scores, n_blocks, block_size):
    scores = np.asarray(scores)
    candidates = []

    for start in range(0, N - block_size + 1, 5):
        candidates.append((
            float(scores[start:start + block_size].mean()),
            start
        ))

    candidates.sort(reverse=True)

    occupied = np.zeros(N, dtype=bool)
    selected = []

    for _, start in candidates:
        end = start + block_size
        if not occupied[start:end].any():
            selected.append(start)
            occupied[start:end] = True
            if len(selected) == n_blocks:
                break

    assert len(selected) == n_blocks
    return selected

def select_random_blocks(n_blocks, block_size, seed):
    local_rng = np.random.default_rng(seed)
    starts = np.arange(N - block_size + 1)
    local_rng.shuffle(starts)

    occupied = np.zeros(N, dtype=bool)
    selected = []

    for start in starts:
        end = start + block_size
        if not occupied[start:end].any():
            selected.append(int(start))
            occupied[start:end] = True
            if len(selected) == n_blocks:
                break

    assert len(selected) == n_blocks
    return selected

def delete_blocks(x, starts, block_size):
    out = np.asarray(x, dtype=np.float64).copy()

    for start in starts:
        end = start + block_size

        left = out[start - 1] if start > 0 else out[end]
        right = out[end] if end < N else left

        out[start:end] = np.linspace(
            left, right, block_size + 2
        )[1:-1]

    return out

def predict_1d(x):
    inp = ((x - MEAN) / SCALE).astype(np.float32)[None, :, None]
    return run_model("1d", m1, inp)[0]

def predict_2d(x, renderer):
    rgb = renderer.pert(x)
    image = (to_2d(rgb[None]) / 255.0).astype(np.float32)
    return run_model("2d", m2, image)[0]

METHODS = {
    "1d": ["R", "H1", "base1"],
    "2d": ["R", "H2", "base2"]
}

records = []
renderer = Renderer()

print("Sensitivity analysis: 40 records, 3 block sizes.")

try:
    for ri, row in sample.iterrows():

        cohort = int(row.cohort_index)
        zi = cohort_to_zi[cohort]
        x = load_ecg(row.ecg_csv_filename)
        target = CLASSES.index(row.class_name)

        assert len(x) == N

        p1 = predict_1d(x)
        rgb0 = renderer.base(x)
        img0 = (to_2d(rgb0[None]) / 255.0).astype(np.float32)
        p2 = run_model("2d", m2, img0)[0]

        assert np.allclose(p1, z["out1_base"][zi], atol=1e-5), (
            f"1D baseline mismatch: {cohort}"
        )
        assert np.allclose(p2, z["out2_base"][zi], atol=1e-5), (
            f"2D baseline mismatch: {cohort}"
        )

        baseline = {"1d": float(p1[target]), "2d": float(p2[target])}

        maps = {
            name: map_to_ecg(z[name][zi])
            for name in ["R", "H1", "H2", "base1", "base2"]
        }

        for branch, methods in METHODS.items():

            for block_size in BLOCK_SIZES:
                for fraction in FRACTIONS:

                    n_blocks = max(
                        1,
                        int(round(fraction * N / block_size))
                    )
                    actual_fraction = n_blocks * block_size / N

                    random_sets = [
                        select_random_blocks(
                            n_blocks,
                            block_size,
                            SEED + cohort * 100000 +
                            block_size * 100 + int(fraction * 100) +
                            rep
                        )
                        for rep in range(RANDOM_REPEATS)
                    ]

                    random_drops = []

                    for starts in random_sets:
                        xr = delete_blocks(x, starts, block_size)

                        if branch == "1d":
                            pr = predict_1d(xr)[target]
                        else:
                            pr = predict_2d(xr, renderer)[target]

                        random_drops.append(
                            baseline[branch] - float(pr)
                        )

                    for method in methods:

                        top_starts = select_top_blocks(
                            maps[method], n_blocks, block_size
                        )
                        xtop = delete_blocks(x, top_starts, block_size)

                        if branch == "1d":
                            pt = predict_1d(xtop)[target]
                        else:
                            pt = predict_2d(xtop, renderer)[target]

                        attr_drop = baseline[branch] - float(pt)

                        records.append({
                            "cohort_index": cohort,
                            "class_name": row.class_name,
                            "branch": branch,
                            "method": method,
                            "block_size": block_size,
                            "requested_fraction": fraction,
                            "actual_fraction": actual_fraction,
                            "attribution_drop": attr_drop,
                            "random_drop_mean": float(np.mean(random_drops)),
                            "paired_difference": (
                                attr_drop - float(np.mean(random_drops))
                            ),
                            "random_drops": ";".join(
                                map(str, random_drops)
                            )
                        })

        if (ri + 1) % 5 == 0:
            print(f"Processed {ri + 1}/{len(sample)} records")
            gc.collect()

finally:
    plt.close(renderer.fig)
    gc.collect()

raw = pd.DataFrame(records)
raw_path = os.path.join(SENS_DIR, "sensitivity_raw.csv")
raw.to_csv(raw_path, index=False)

def boot_ci(values, seed):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 2:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    draws = rng.choice(
        values, size=(BOOTSTRAPS, len(values)), replace=True
    ).mean(axis=1)

    return (
        float(values.mean()),
        float(np.quantile(draws, 0.025)),
        float(np.quantile(draws, 0.975))
    )

summary_rows = []

for (branch, method, block_size, fraction), g in raw.groupby(
    ["branch", "method", "block_size", "requested_fraction"]
):
    mean, low, high = boot_ci(
        g["paired_difference"], SEED + block_size
    )

    attr_mean, attr_low, attr_high = boot_ci(
        g["attribution_drop"], SEED + 1
    )

    random_mean, random_low, random_high = boot_ci(
        g["random_drop_mean"], SEED + 2
    )

    summary_rows.append({
        "branch": branch,
        "method": method,
        "block_size": block_size,
        "fraction": fraction,
        "n_records": len(g),
        "attribution_mean_drop": attr_mean,
        "random_mean_drop": random_mean,
        "paired_difference": mean,
        "difference_ci_low": low,
        "difference_ci_high": high
    })

summary = pd.DataFrame(summary_rows)
summary_path = os.path.join(SENS_DIR, "sensitivity_summary.csv")
summary.to_csv(summary_path, index=False)

print("\nSENSITIVITY SUMMARY")
display(summary.round(4))

for branch in ["1d", "2d"]:
    fig, ax = plt.subplots(figsize=(9, 6))

    sub = summary[
        (summary.branch == branch) &
        (summary.method == "R")
    ]

    for block_size in BLOCK_SIZES:
        s = sub[sub.block_size == block_size].sort_values("fraction")

        ax.plot(
            s.fraction * 100,
            s.attribution_mean_drop,
            marker="o",
            label=f"R attribution, block={block_size}"
        )

        ax.plot(
            s.fraction * 100,
            s.random_mean_drop,
            marker="s",
            linestyle="--",
            label=f"Random, block={block_size}"
        )

    ax.axhline(0, linewidth=0.8)
    ax.set_xlabel("ECG signal deleted (%)")
    ax.set_ylabel("Drop in true-class probability")
    ax.set_title(f"Deletion Sensitivity — {branch.upper()}")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    fig.savefig(
        os.path.join(SENS_DIR, f"sensitivity_{branch}.png"),
        dpi=200
    )
    plt.show()
    plt.close(fig)

print("\nCompleted.")
print("Raw:", raw_path)
print("Summary:", summary_path)
print("Output directory:", SENS_DIR)


## Final paper figures

In [ ]:
# ============================================================

import os
import glob
import zipfile
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, FileLink

ROOT = "/kaggle/working/m2_out"

SENS_DIR = os.path.join(ROOT, "deletion_sensitivity_v1")
FAITH_DIR = os.path.join(ROOT, "faithfulness_corrected_v1")
MAP_DIR = os.path.join(ROOT, "mapping_audit_v1")

OUT = os.path.join(ROOT, "module2_paper_figures")
os.makedirs(OUT, exist_ok=True)

sens_path = os.path.join(SENS_DIR, "sensitivity_summary.csv")
map_path_candidates = glob.glob(os.path.join(MAP_DIR, "*.csv"))

assert os.path.exists(sens_path), f"Missing: {sens_path}"
assert map_path_candidates, "Mapping audit CSV not found."

sens = pd.read_csv(sens_path)
mapping = pd.read_csv(map_path_candidates[0])

plt.rcParams.update({
    "figure.figsize": (9, 5.5),
    "figure.dpi": 150,
    "font.size": 11,
    "axes.titlesize": 15,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "savefig.bbox": "tight"
})

COLORS = {
    "R": "#D46A94",
    "H1": "#4D94B7",
    "H2": "#82A65B",
    "base1": "#AE8266",
    "base2": "#9670B8"
}

def savefig(name):
    plt.savefig(os.path.join(OUT, name), dpi=300)
    plt.close()

required = {
    "map",
    "mean_mass_outside_geometry",
    "mean_top10_percent_outside_geometry"
}
assert required.issubset(mapping.columns), (
    f"Unexpected mapping columns: {mapping.columns.tolist()}"
)

m = mapping.copy()

plt.figure()
x = np.arange(len(m))
w = 0.36

plt.bar(
    x - w/2,
    m["mean_mass_outside_geometry"],
    w,
    label="Mean attribution mass outside ECG region"
)
plt.bar(
    x + w/2,
    m["mean_top10_percent_outside_geometry"],
    w,
    label="Top-10% attribution outside ECG region"
)

plt.xticks(x, m["map"])
plt.ylabel("Attribution proportion")
plt.title("Module 2 — Attribution-to-signal mapping audit")
plt.legend()
savefig("01_mapping_audit.png")

required = {
    "branch", "method", "block_size", "fraction",
    "paired_difference"
}
assert required.issubset(sens.columns), (
    f"Unexpected sensitivity columns: {sens.columns.tolist()}"
)

def plot_sensitivity(branch, filename, title):
    d = sens[sens["branch"].str.lower() == branch].copy()

    plt.figure(figsize=(9, 5.5))

    methods = list(d["method"].dropna().unique())

    for method in methods:
        for block in sorted(d["block_size"].unique()):
            q = d[
                (d["method"] == method) &
                (d["block_size"] == block)
            ].sort_values("fraction")

            if q.empty:
                continue

            x = q["fraction"].to_numpy() * 100
            y = q["paired_difference"].to_numpy()

            plt.plot(
                x, y, marker="o",
                label=f"{method}, block={block}"
            )

    plt.axhline(0, color="black", linewidth=1)
    plt.xlabel("ECG signal deleted (%)")
    plt.ylabel("Attribution drop − random drop")
    plt.title(title)
    plt.legend(fontsize=8, ncol=2)
    savefig(filename)

plot_sensitivity(
    "1d",
    "02_deletion_sensitivity_1d.png",
    "Module 2 — 1D deletion sensitivity"
)

plot_sensitivity(
    "2d",
    "03_deletion_sensitivity_2d.png",
    "Module 2 — 2D deletion sensitivity"
)

low_col = next(
    (c for c in sens.columns if c in
     ["difference_ci_low", "diff_ci_low", "ci_low"]),
    None
)
high_col = next(
    (c for c in sens.columns if c in
     ["difference_ci_high", "diff_ci_high", "ci_high"]),
    None
)

if low_col and high_col:
    for branch in ["1d", "2d"]:
        d = sens[sens["branch"].str.lower() == branch].copy()
        d = d.sort_values(["method", "block_size", "fraction"])

        if d.empty:
            continue

        labels = (
            d["method"].astype(str) +
            " | " + d["block_size"].astype(str) +
            " | " + (d["fraction"] * 100).round().astype(int).astype(str) + "%"
        )

        y = d["paired_difference"].to_numpy()
        lo = d[low_col].to_numpy()
        hi = d[high_col].to_numpy()

        plt.figure(figsize=(10, max(5, len(d) * 0.24)))
        positions = np.arange(len(d))

        plt.errorbar(
            y, positions,
            xerr=[y - lo, hi - y],
            fmt="o",
            capsize=2
        )

        plt.axvline(0, color="black", linewidth=1)
        plt.yticks(positions, labels, fontsize=7)
        plt.xlabel("Paired difference (attribution − random)")
        plt.title(f"Module 2 — {branch.upper()} paired differences with confidence intervals")
        savefig(f"04_paired_ci_{branch}.png")

plt.figure(figsize=(8, 5))

plt.bar(
    mapping["map"],
    mapping["mean_mass_outside_geometry"]
)

plt.ylabel("Mean attribution mass outside ECG region")
plt.title("Module 2 — Attribution outside valid ECG geometry")
savefig("05_outside_geometry.png")

for folder in [FAITH_DIR, SENS_DIR, MAP_DIR]:
    if os.path.isdir(folder):
        for file in glob.glob(os.path.join(folder, "*.csv")):
            target = os.path.join(OUT, os.path.basename(file))
            if not os.path.exists(target):
                import shutil
                shutil.copy2(file, target)

zip_path = os.path.join(ROOT, "module2_paper_figures.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in glob.glob(os.path.join(OUT, "*")):
        z.write(file, arcname=os.path.basename(file))

print("\nCompleted.")
print("Figures and conclusion:", OUT)
print("ZIP:", zip_path)

display(FileLink(zip_path))
